# Session 8 · Build, deploy, update, break, and move

Queen City Trip Analytics, a fictional South End, Charlotte company, ships its fare-quote API to the fleets that call it.

In [ ]:
source ~/dsba6190-live-demo-08/env.sh && echo "$CLUSTER   $IMAGE_BASE" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · The Dockerfile, and the layer cache

In [ ]:
cat app/Dockerfile

In [ ]:
docker build --progress=plain -t fare-api:local app 2>&1 | grep -E 'CACHED|DONE|\[[0-9]/[0-9]\]' | tail -12

In [ ]:
echo "# a one-line change" >> app/main.py
docker build --progress=plain -t fare-api:local app 2>&1 | grep -E 'CACHED|DONE|\[[0-9]/[0-9]\]' | tail -12

## Step 2 · Cloud Build, Artifact Registry, and the scan

In [ ]:
rm -rf build-live; cp -R app build-live; echo v1 > build-live/VERSION
gcloud builds submit build-live --region "$REGION" --tag "$IMAGE_BASE:v1"

In [ ]:
gcloud artifacts docker images list "$REGION-docker.pkg.dev/$PROJECT/$REPO" --include-tags

In [ ]:
gcloud artifacts docker images describe "$IMAGE_BASE:v2" --show-package-vulnerability \
  --format='yaml(package_vulnerability_summary.vulnerabilities)' | grep -E '^ *[A-Z]+:' 

## Step 3 · Three replicas and a Service

In [ ]:
kubectl apply -f manifests/deployment-v1.yaml -f manifests/service.yaml

In [ ]:
kubectl rollout status deployment/fare-api

In [ ]:
kubectl get pods -l app=fare-api -o wide

In [ ]:
for i in $(seq 1 30); do IP=$(kubectl get service fare-api -o jsonpath="{.status.loadBalancer.ingress[0].ip}"); [ -n "$IP" ] && break; sleep 5; done; echo "external IP: $IP"

In [ ]:
IP=$(kubectl get service fare-api -o jsonpath="{.status.loadBalancer.ingress[0].ip}")
curl -s "http://$IP/quote?pickup=132&dropoff=236"; echo

## Step 4 · Delete a pod

In [ ]:
kubectl delete pod "$(kubectl get pods -l app=fare-api -o jsonpath='{.items[0].metadata.name}')" --wait=false
sleep 3; kubectl get pods -l app=fare-api

In [ ]:
kubectl get pods -l app=fare-api

## Step 5 · Scale out and in while the dispatcher keeps asking

In [ ]:
./loop.sh > loop.log 2>&1 &
echo $! > loop.pid; echo "loop started"

In [ ]:
kubectl scale deployment fare-api --replicas=6
kubectl rollout status deployment/fare-api
kubectl get pods -l app=fare-api

In [ ]:
kubectl scale deployment fare-api --replicas=3

In [ ]:
tail -6 loop.log

## Step 6 · Rolling update to v2, then undo

In [ ]:
kubectl apply -f manifests/deployment-v2.yaml
kubectl rollout status deployment/fare-api

In [ ]:
tail -6 loop.log

In [ ]:
kubectl rollout history deployment/fare-api

In [ ]:
kubectl rollout undo deployment/fare-api
kubectl rollout status deployment/fare-api

In [ ]:
tail -6 loop.log

## Step 7 · Probes: none, wrong, right

In [ ]:
kubectl apply -f manifests/deployment-noprobe.yaml
kubectl rollout status deployment/fare-api

In [ ]:
grep ERR loop.log | tail -8; grep -c ERR loop.log

In [ ]:
kubectl apply -f manifests/deployment-badprobe.yaml
sleep 30; kubectl get pods -l app=fare-api

In [ ]:
kubectl describe pod "$(kubectl get pods -l app=fare-api -o jsonpath='{range .items[?(@.status.containerStatuses[0].ready==false)]}{.metadata.name}{"\n"}{end}' | head -1)" \
  | sed -n '/Events:/,$p' | tail -6

In [ ]:
kubectl rollout status deployment/fare-api --timeout=20s

In [ ]:
kubectl apply -f manifests/deployment-goodprobe.yaml
kubectl rollout status deployment/fare-api --timeout=300s

In [ ]:
kill "$(cat loop.pid)" 2>/dev/null; sleep 1; grep -c " 200 " loop.log; grep -c " ERR " loop.log

## Step 8 · Two limits, two failures

In [ ]:
kubectl apply -f manifests/deployment-oom.yaml
sleep 40; kubectl get pods -l app=fare-api-oom

In [ ]:
kubectl describe pod -l app=fare-api-oom | grep -A6 'Last State'

In [ ]:
kubectl apply -f manifests/deployment-throttled.yaml
kubectl rollout status deployment/fare-api-throttled

In [ ]:
for i in 1 2 3; do kubectl exec deploy/fare-api -- python -c 'import json,urllib.request;print(json.load(urllib.request.urlopen("http://localhost:8080/quote"))["ms"],"ms")'; done

In [ ]:
for i in 1 2 3; do kubectl exec deploy/fare-api-throttled -- python -c 'import json,urllib.request;print(json.load(urllib.request.urlopen("http://localhost:8080/quote"))["ms"],"ms")'; done

In [ ]:
kubectl get pods -l app=fare-api-throttled
kubectl delete deployment fare-api-oom fare-api-throttled --wait=false

## Step 9 · A disruption budget, then a node drain

In [ ]:
NODE=$(kubectl get pods -l app=fare-api -o jsonpath='{.items[0].spec.nodeName}'); echo "$NODE"
kubectl apply -f manifests/pdb-strict.yaml

In [ ]:
kubectl drain "$NODE" --ignore-daemonsets --delete-emptydir-data --pod-selector=app=fare-api --timeout=40s 2>&1 | grep -v "^WARNING" | tail -5

In [ ]:
kubectl apply -f manifests/pdb.yaml
kubectl get pdb fare-api

In [ ]:
kubectl drain "$NODE" --ignore-daemonsets --delete-emptydir-data --pod-selector=app=fare-api --timeout=120s 2>&1 | grep -v "^WARNING" | tail -5

In [ ]:
kubectl get pods -l app=fare-api -o wide

In [ ]:
kubectl uncordon "$NODE"

## Step 10 · The same image on Cloud Run

In [ ]:
time gcloud run deploy qc-fare-api --region "$REGION" --image "$IMAGE_BASE:v2" --allow-unauthenticated --quiet

In [ ]:
URL=$(gcloud run services describe qc-fare-api --region "$REGION" --format="value(status.url)")
for i in 1 2 3; do curl -s -w "  %{time_total}s" "$URL/quote?pickup=132&dropoff=236"; echo; done

## Step 11 · Teardown, in order

In [ ]:
kubectl delete service fare-api --wait=true
gcloud compute forwarding-rules list

In [ ]:
kubectl delete deployment fare-api; kubectl delete pdb fare-api

In [ ]:
gcloud container clusters delete "${CLUSTER:?}" --zone "${ZONE:?}" --quiet

In [ ]:
gcloud run services delete qc-fare-api --region "${REGION:?}" --quiet
gcloud artifacts repositories delete "${REPO:?}" --location "${REGION:?}" --quiet

In [ ]:
gcloud container clusters list; gcloud run services list --region "$REGION"; gcloud compute forwarding-rules list